# 실전 — 실제 데이터로 검증: CICIDS2017 로 검증하기 (79-feature flow + 실제 데이터 품질)

이 책의 대표 데이터셋 **CICIDS2017**(CICFlowMeter 79 feature) 의 실제 파일을 내려받아, ch08 진단 → ch09 정제(inf·NaN) → ch14 지도학습 파이프라인을 그대로 돌린다. 합성 데이터와 달리 **실제 inf·결측·인코딩 깨짐** 이 들어 있어, ch08~09 이 왜 데이터 품질을 강조하는지 실물로 드러난다.

> **인터넷 필요**: 실행 시 CICIDS2017 의 하루치 파일(Web 공격, 약 49MB)을 내려받는다.

핵심 관찰: (1) 실제 CICIDS 파일에는 컬럼명 앞뒤 공백, `Flow Bytes/s` 의 inf(0으로 나눔), 결측, 라벨 인코딩 깨짐이 있다 — ch09 의 정제가 전처리 첫 단계다. (2) 이진 탐지는 거의 완벽하지만, **SQL Injection 은 전체 21건뿐이라 recall 이 0.1~0.2 로 사실상 못 잡힌다.** 정확도 0.996 뒤에 숨은 이 실패를 macro-F1 이 드러낸다(ch16).

## 데이터셋 이야기 — 어떻게 수집됐나 (CICIDS2017)

캐나다 사이버보안연구소(CIC)가 2017년 **5일간(월~금) 구축한 테스트베드**에서 나왔다. 25명 사용자의 HTTP·HTTPS·FTP·SSH·이메일 행동을 프로파일링한 **실제 정상 트래픽** 위에, 요일별로 브루트포스·DoS·DDoS·웹 공격·침투·봇넷·포트스캔을 **실제로 실행**해 pcap 을 캡처하고 CICFlowMeter 로 79개 flow feature 를 추출했다.

**시나리오**: 현실적인 기업 트래픽과 라벨된 실제 공격이 하루 단위로 섞인 로그에서 공격을 잡아낼 수 있는가.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# !pip install pandas scikit-learn matplotlib

from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

import urllib.request, ssl, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

## 1. 실데이터 내려받기 (CICIDS2017 — Web 공격 하루치)

BENIGN 과 웹 공격(Brute Force·XSS·SQL Injection) 이 섞인 실제 flow 파일을 받는다.

In [ ]:
BASE = 'https://huggingface.co/datasets/c01dsnap/CIC-IDS2017/resolve/main/'
FNAME = 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
def fetch(local, url):
    if not os.path.exists(local):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=180, context=ssl.create_default_context()) as r:
            open(local, 'wb').write(r.read())
    return local

df = pd.read_csv(fetch('cicids_web.csv', BASE + FNAME.replace(' ', '%20')), low_memory=False)
df.columns = df.columns.str.strip()          # ch08: CICIDS 는 컬럼명에 공백이 있다
print('shape:', df.shape)
print(df['Label'].value_counts())

## 2. ch08 진단 → ch09 정제 (실제 데이터 품질 문제)

실제 CICIDS 는 `Flow Bytes/s` 등에서 0으로 나눠 생긴 **inf**, 소수의 **결측**, 라벨의 **인코딩 깨짐**(`Web Attack � XSS`) 을 갖는다. inf 를 NaN 으로 바꾸고 중앙값으로 대치하는 것이 ch09 의 표준 처리다.

In [ ]:
# 라벨 인코딩 깨짐 정리
df['Label'] = (df['Label'].str.replace('Web Attack.*Brute Force', 'BruteForce', regex=True)
                          .str.replace('Web Attack.*XSS', 'XSS', regex=True)
                          .str.replace('Web Attack.*Sql Injection', 'SQLi', regex=True))

X = df.drop(columns=['Label']).select_dtypes(include=[np.number])
n_inf = int(np.isinf(X.to_numpy()).sum())
X = X.replace([np.inf, -np.inf], np.nan)      # ch09: inf → NaN
print(f'inf {n_inf} 건 → NaN 으로, 결측 포함 컬럼 {int(X.isna().any().sum())}개는 median 대치 예정')
y_bin = (df['Label'] != 'BENIGN').astype(int)
y_type = df['Label']

## 3. 이진 탐지 (BENIGN vs 웹공격)

In [ ]:
# ── 실습 과제 ──
# 목표: 3. 이진 탐지 (BENIGN vs 웹공격)
# 만들 결과: clf, proba, pred
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
Xtr, Xte, ytr, yte, mtr, mte = ...
    X, y_bin, y_type, test_size = ...

clf = ...
    ('impute', SimpleImputer(strategy = ...
    ...
    ('rf', RandomForestClassifier(n_estimators = ...
...

proba = ...
pred = ...
print(classification_report(yte, pred, target_names = ...
...
```
</details>


## 4. 공격 유형 분류 — SQL Injection 21건의 벽

이진 탐지는 거의 완벽하지만, **SQL Injection 은 전체 21건**뿐이라 테스트에는 6건만 들어간다. `class_weight='balanced'` 로도 recall 은 0.1~0.2 다. **정확도 0.996 은 이 실패를 완전히 가린다** — macro-F1 을 봐야 보인다.

In [ ]:
# ── 실습 과제 ──
# 목표: 4. 공격 유형 분류 — SQL Injection 21건의 벽 — 이진 탐지는 거의 완벽하지만, **SQL Injection 은 전체 21건**뿐이라 테스트에는 6건만 들어간다. `class_weight='balanced'` 로도 recall 은 0.1~0.2 다. **정확도 0.996
# 만들 결과: clf_m, class_weight
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
clf_m = ...
    ('impute', SimpleImputer(strategy = ...
    ...
    ('rf', RandomForestClassifier(n_estimators = ...
                                  class_weight = ...
...

print(classification_report(mte, clf_m.predict(Xte), digits = ...
```
</details>


## 5. 정리 — 실 CICIDS2017 에서 확인한 것

1. **실데이터는 지저분하다.** 컬럼명 공백·inf·결측·라벨 인코딩 깨짐을 ch09 정제 없이 넣으면 학습이 즉시 실패한다. 이것이 합성 데이터에는 없는, 실무의 첫 장벽이다.
2. **정확도는 거짓말을 한다.** 이진·다중 모두 정확도 99% 대지만, **SQL Injection(21건)의 recall 은 0.1~0.2** 다. 표본이 극단적으로 적은 공격은 실데이터에서 사실상 탐지 불가에 가깝다.
3. **그래서 클래스별 recall·macro-F1 이 필수다(ch16).** 그리고 이런 희소 공격은 지도학습만으로는 한계가 있어, 비지도 이상 탐지(ch11)·페이로드 NLP(ch15)·데이터 보강(ch08) 과 결합해야 한다.

> 같은 코드로 CICIDS2017 의 다른 날짜 파일(DDoS·PortScan·Infiltration)로 바꿔 실행할 수 있다. `FNAME` 만 교체하면 된다.